# Baseline 1 — Persistence

**Idea:** the forecast for each 15-minute block of day D+1 is the actual power at the same block on day D ("tomorrow = today").

No training and nothing to tune. Every later model (MLR, Ridge, Random Forest, AdaBoost) must beat this score.

Input: `Data/Clean/Winter_Clean.csv` (15 Oct 2025 – 14 Feb 2026, 96 blocks per day).

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 1. Settings

In [ ]:
# Works whether the notebook is opened from src/ or from the project folder.
ROOT = Path.cwd().parent if Path.cwd().name == "src" else Path.cwd()
CLEAN_CSV = ROOT / "Data" / "Clean" / "Winter_Clean.csv"
RESULTS = ROOT / "results"

POWER = "Act. Power (MW)"

CAPACITY_MW = 21.5
BAND_MW = 0.07 * CAPACITY_MW  # ±7% of capacity (GERC 2019 tolerance) = 1.505 MW

# Scoring window 07:00-18:15 (blocks 29-74): the plant never generates outside it.
DAY_FIRST_BLOCK, DAY_LAST_BLOCK = 29, 74

# Chronological split by target day (dates inclusive).
SPLITS = {
    "train": ("2025-10-15", "2026-01-14"),
    "val": ("2026-01-15", "2026-02-07"),
    "test": ("2026-02-08", "2026-02-14"),
}

## 2. Load the clean data and label each day with its split

In [ ]:
df = pd.read_csv(CLEAN_CSV, parse_dates=["TimeStamp", "Date"])

df["split"] = None
for name, (start, end) in SPLITS.items():
    df.loc[df["Date"].between(start, end), "split"] = name
assert df["split"].notna().all(), "some dates fall outside every split"

print(df.shape)
df.groupby("split")["Date"].agg(["min", "max", "nunique"]).loc[["train", "val", "test"]]

## 3. Persistence forecast: same block, previous day

In [ ]:
df["forecast"] = df.groupby("Block")[POWER].shift(1)

# Example: block 46 (11:15) — each day's forecast is the previous day's actual.
df.loc[df["Block"] == 46, ["Date", POWER, "forecast"]].head()

In [ ]:
# 15 Oct has no previous day, so it has no forecast.
df = df.dropna(subset=["forecast"]).rename(columns={POWER: "actual"})
print(df.shape)

## 4. Metrics

In [ ]:
def evaluate(actual, forecast):
    err = forecast - actual
    mae = err.abs().mean()
    rmse = np.sqrt((err**2).mean())
    # R² = 1 - SS_res / SS_tot: share of variation explained vs always predicting the mean.
    r2 = 1 - (err**2).sum() / ((actual - actual.mean()) ** 2).sum()
    return {
        "MAE (MW)": mae,
        "RMSE (MW)": rmse,
        "nMAE (%)": 100 * mae / CAPACITY_MW,
        "nRMSE (%)": 100 * rmse / CAPACITY_MW,
        "Within ±7% (%)": 100 * (err.abs() <= BAND_MW).mean(),
        "R² (%)": 100 * r2,
        "Blocks": len(err),
    }

## 5. Score each split on daytime blocks (07:00–18:15)

In [ ]:
daytime = df[df["Block"].between(DAY_FIRST_BLOCK, DAY_LAST_BLOCK)]

rows = []
for split in ["train", "val", "test"]:
    part = daytime[daytime["split"] == split]
    rows.append({"split": split, "days": part["Date"].nunique(), **evaluate(part["actual"], part["forecast"])})

metrics = pd.DataFrame(rows).round(3)
metrics

## 6. Where does persistence fail? Worst validation days

In [ ]:
daily_mae = (
    daytime[daytime["split"] == "val"]
    .assign(abs_err=lambda x: (x["forecast"] - x["actual"]).abs())
    .groupby("Date")["abs_err"]
    .mean()
    .round(2)
)
daily_mae.sort_values(ascending=False).head(5)

In [ ]:
# A clear day, then a cloudy day, then a clear day: persistence is always one day late.
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, day in zip(axes, ["2026-01-25", "2026-01-26", "2026-01-27"]):
    d = df[df["Date"] == day]
    hours = (d["Block"] - 1) / 4
    ax.fill_between(hours, d["actual"] - BAND_MW, d["actual"] + BAND_MW, alpha=0.2, label="±7% band")
    ax.plot(hours, d["actual"], label="Actual")
    ax.plot(hours, d["forecast"], "--", label="Persistence")
    ax.set_title(day)
    ax.set_xlabel("Hour of day")
    ax.set_xlim(6, 19)
axes[0].set_ylabel("Power (MW)")
axes[0].legend()
plt.tight_layout()
plt.show()

## 7. Save forecasts and scores

In [ ]:
(RESULTS / "predictions").mkdir(parents=True, exist_ok=True)
(RESULTS / "metrics").mkdir(parents=True, exist_ok=True)

df[["TimeStamp", "Date", "Block", "split", "actual", "forecast"]].to_csv(RESULTS / "predictions" / "persistence.csv", index=False)
metrics.to_csv(RESULTS / "metrics" / "persistence.csv", index=False)